# 025-a-contrario-gate checks

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, frame_place, load_frames
from uavloc.data.match_pairs import village_split
from uavloc.data.multidate import load_date
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.acontrario import (NullModel, OwnMapNull, VillageNull, cropped_drone, nfa, raw_scores, raw_surface,
                                        score_nfa_draws)
from uavloc.matching.benchmark import Draw, benchmark_draws
from uavloc.matching.correlator import PositionCorrelator, class_ncc_maps, z_scored
from uavloc.matching.identity import SatelliteAsDroneCorrelator
from uavloc.track import REPO_ROOT, experiment_dir, load_runs, set_seed

# Params

In [ ]:
EXP = "025-a-contrario-gate"
REFERENCE_EXP = "022-adaptive-gate"
REFERENCE_RUN = "022.2"
REPRODUCTION_RUN = "025.1"
REPRODUCTION_TOL_M = 0.001
NFA_EPSILON = 1.0
IDENTITY_MIN_ACCEPT_SHARE = 0.95
FALSE_ALARM_MAX_SHARE = 0.05
RAW_TOL = 1e-4                    # FFT (batched) vs direct-conv float32 NCC, per cell; measured 6.4e-5 on bushcha @10 m
Z_TOL = 1e-3                      # z-scored raw score vs the bench heatmap, per cell; measured 1.8e-4
WRONG_WINDOWS_PER_FRAME = 8
WRONG_SEED = 42
NULL_WINDOWS = 200
NULL_SEED = 42
OWN_MAP_EXCLUSION_M = 60.0
NFA_PLOT_FLOOR = 1e-3

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
FRAMES_DATASET = "village-crossdate-frames"
EXCLUDE_PLACES = ["birky-49p9105-23p9192", "bushcha-50p2422-25p8305", "dubrova-49p5175-24p0765"]
N_VAL_VILLAGES = 10
SPLIT_SEED = 42
MAP_DATE = "newest"
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
TORCH_THREADS = 2
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
SAT_FEATURE_CROP_PCT = 0.03
RUN_CONFIG = {"method": "segmentation", "matching": "weighted-ncc", "class_weights": (0.0, 1.0, 1.0, 0.5, 0.0, 0.5),
              "window": "rect", "search_scale": 1.7, "search_radius_m": None, "prior_sigma_m": None, "gate": "none",
              "fov_w_deg": 60.0, "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": 0.0}

# Data

In [ ]:
set_seed(SEED)
torch.set_num_threads(TORCH_THREADS)
train_villages, val_villages = village_split(EXCLUDE_PLACES, N_VAL_VILLAGES, SPLIT_SEED)
frames = load_frames(FRAMES_DATASET)
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
{place: len(m) for place, m in places.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)
config = correlator_config(RUN_CONFIG)

In [ ]:
def map_features(place: str) -> SegmentationFeatures:
    return SegmentationFeatures(model, model, load_date(place, MAP_DATE), SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)

In [ ]:
features = {place: map_features(place) for place in PLACES}
correlators = {place: PositionCorrelator(features[place], config) for place in PLACES}
identity = {place: SatelliteAsDroneCorrelator(features[place], config) for place in PLACES}
patch_size_m = correlators[PLACES[0]].patch_size_m(places[PLACES[0]][0])
patch_size_m

In [ ]:
village_null = VillageNull.build(train_villages, map_features, patch_size_m, SAT_FEATURE_CROP_PCT, NULL_WINDOWS, NULL_SEED)

In [ ]:
n_wrong = sum(len(m) for m in places.values()) * WRONG_WINDOWS_PER_FRAME
wrong_bank = VillageNull.build(val_villages, map_features, patch_size_m, SAT_FEATURE_CROP_PCT, n_wrong, WRONG_SEED)
assert not {p.place for p in wrong_bank.poses} & (set(train_villages) | set(PLACES))

# Run

In [ ]:
def null_models() -> dict[str, NullModel]:
    """Both pre-registered nulls, caches empty (they key on frame names, and a check swaps the drone map)."""
    village_null.clear_cache()
    return {"villages": village_null, "own-map": OwnMapNull(NULL_WINDOWS, OWN_MAP_EXCLUSION_M, NULL_SEED)}

In [ ]:
def literal_raw(correlator: PositionCorrelator, draw: Draw) -> np.ndarray:
    """The proposal's raw score verbatim: `class_ncc_maps` weighted as `weighted_ncc_heatmap`, before z-scoring."""
    drone, satellite, _, _ = correlator.prepare(correlator.drone_map(draw.frame), draw.prior, draw.frame.yaw_deg,
                                                correlator.patch_size_m(draw.frame))
    maps = class_ncc_maps(drone, satellite, config.class_weights)
    if not maps:
        return np.zeros(draw.surface.heatmap.shape, np.float32)
    return sum(config.class_weights[c] * m for c, m in maps.items()) / sum(config.class_weights[c] for c in maps)

In [ ]:
raw_draws = benchmark_draws(correlators[PLACES[0]], places[PLACES[0]], PRIMARY_NOISE_M, N_DIRECTIONS, SEED)
raw_rows = []
for draw in tqdm(raw_draws, desc="raw score"):
    raw = raw_surface(correlators[PLACES[0]], draw.frame, draw.prior)
    flat = raw.max() == raw.min()
    raw_rows.append({"raw_diff": float(np.abs(raw - literal_raw(correlators[PLACES[0]], draw)).max()),
                     "z_diff": 0.0 if flat else float(np.abs(z_scored(torch.from_numpy(raw)) - draw.surface.heatmap).max()),
                     "same_argmax": flat or np.argmax(raw) == np.argmax(draw.surface.heatmap)})
raw_check = pd.DataFrame(raw_rows)
raw_check.describe()

In [ ]:
identity_parts = []
for place in PLACES:
    draws = [d for noise_m in NOISE_M for d in benchmark_draws(identity[place], places[place], noise_m, N_DIRECTIONS, SEED)]
    for name, null in null_models().items():
        identity_parts.append(score_nfa_draws(draws, identity[place], null, NFA_EPSILON).assign(place=place, null=name))
identity_records = pd.concat(identity_parts, ignore_index=True)
reachable = identity_records[identity_records["reachable"] & (identity_records["status"] == "ok")]
identity_share = (reachable["nfa"] < NFA_EPSILON).groupby(reachable["null"]).mean()
identity_share

In [ ]:
blind = reachable["raw_peak"] == 0   # no weighted class in the query: raw surface all zero, NFA = N_tests
identity_breakdown = (reachable["nfa"] < NFA_EPSILON).groupby([reachable["null"], blind.rename("blind")]).agg(["mean", "size"])
identity_breakdown

In [ ]:
def false_alarm_rows(place: str, nulls: dict[str, NullModel], first_window: int) -> list[dict]:
    """Each frame's drone map against `WRONG_WINDOWS_PER_FRAME` windows of another village, scored as if true."""
    correlator, rows = correlators[place], []
    for i, frame in enumerate(tqdm(places[place], desc=f"wrong windows {place.split('-')[0]}")):
        drone = cropped_drone(correlator, frame)
        start = first_window + i * WRONG_WINDOWS_PER_FRAME
        windows = wrong_bank.windows[start:start + WRONG_WINDOWS_PER_FRAME]
        for k, raw in enumerate(raw_scores(drone, windows, config.class_weights)):
            for name, null in nulls.items():
                pool = null.pool(correlator, frame, frame.position)
                rows.append({"place": place, "frame": frame.name, "window": start + k, "null": name,
                             "wrong_village": wrong_bank.poses[start + k].place, "raw_peak": float(raw.max()),
                             "nfa": nfa(float(raw.max()), raw.size, pool)})
    return rows

In [ ]:
nulls = null_models()
first_windows = np.cumsum([0] + [len(places[p]) * WRONG_WINDOWS_PER_FRAME for p in PLACES[:-1]])
false_alarm = pd.DataFrame([row for place, first in zip(PLACES, first_windows)
                            for row in false_alarm_rows(place, nulls, int(first))])
false_alarm_share = (false_alarm["nfa"] < NFA_EPSILON).groupby(false_alarm["null"]).mean()
false_alarm_share

In [ ]:
def run_record(exp: str, run: str) -> dict:
    return next(r for r in load_runs(exp) if r["run"] == run)

In [ ]:
reference = run_record(REFERENCE_EXP, REFERENCE_RUN)["metrics"]
reproduction = run_record(EXP, REPRODUCTION_RUN)["metrics"]
artifacts = experiment_dir(EXP) / "artifacts"
reference_records = pd.read_csv(experiment_dir(REFERENCE_EXP) / "artifacts" / f"records-{REFERENCE_RUN}.csv")
reproduction_records = pd.read_csv(artifacts / f"records-{REPRODUCTION_RUN}.csv")
keys = ["frame", "noise_m", "bearing_deg"]
paired = reference_records.merge(reproduction_records, on=keys, suffixes=("_ref", ""))
reproduction_check = {
    "mean_ref": reference["mean_position_error_m"], "mean": reproduction["mean_position_error_m"],
    "harm_ref": reference["harm_share"], "harm": reproduction["harm_share"],
    "accept_ref": reference["accept_share"], "accept": reproduction["accept_share"],
    "rows_paired": len(paired), "rows_ref": len(reference_records),
    "max_row_error_diff_m": float((paired["error_m"] - paired["error_m_ref"]).abs().max()),
    "decisions_equal": bool((paired["decision"] == paired["decision_ref"]).all())}
reproduction_check

# Plots

In [ ]:
fig_nfa, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, name in zip(axes, ("villages", "own-map")):
    for label, table in (("identity (true window)", reachable), ("wrong village", false_alarm)):
        values = np.log10(np.maximum(table.loc[table["null"] == name, "nfa"], NFA_PLOT_FLOOR))
        ax.hist(values, bins=40, alpha=.5, density=True, label=label)
    ax.axvline(np.log10(NFA_EPSILON), color="k", ls="--", lw=1)
    ax.set_title(f"null: {name}"); ax.set_xlabel(f"log10 NFA (floored at {NFA_PLOT_FLOOR:g})"); ax.legend(); ax.grid(alpha=.3)
axes[0].set_ylabel("density")
fig_nfa.tight_layout()
(experiment_dir(EXP) / "figures").mkdir(exist_ok=True)
fig_nfa.savefig(experiment_dir(EXP) / "figures" / "checks-nfa-identity-vs-wrong.png", dpi=120)

# Checks

In [ ]:
results = {
    "a_identity_accept_share": identity_share.to_dict(), "a_identity_reachable_draws": int(len(reachable) / 2),
    "a_identity_blind_draws": int(blind.sum() / 2),
    "a_identity_accept_share_non_blind": (reachable.loc[~blind, "nfa"] < NFA_EPSILON).groupby(reachable.loc[~blind, "null"]).mean().to_dict(),
    "a_identity_blind_frames": sorted(reachable.loc[blind, "frame"].unique()),
    "b_false_alarm_share": false_alarm_share.to_dict(), "b_wrong_windows": int(len(false_alarm) / 2),
    "c_reproduction": reproduction_check,
    "d_raw_max_diff": float(raw_check["raw_diff"].max()), "d_z_max_diff": float(raw_check["z_diff"].max()),
    "d_same_argmax_share": float(raw_check["same_argmax"].mean()),
}
passed = {
    # amended 2026-10-20 (proposal.md): reachable non-blind draws; the all-reachable share stays logged in results
    "a": bool((pd.Series(results["a_identity_accept_share_non_blind"]) >= IDENTITY_MIN_ACCEPT_SHARE).all()),
    "b": bool((false_alarm_share <= FALSE_ALARM_MAX_SHARE).all()),
    "c": abs(reproduction_check["mean"] - reproduction_check["mean_ref"]) <= REPRODUCTION_TOL_M
         and reproduction_check["harm"] == reproduction_check["harm_ref"],
    "d": results["d_raw_max_diff"] <= RAW_TOL and results["d_z_max_diff"] <= Z_TOL and results["d_same_argmax_share"] == 1.0,
}
(artifacts / "checks.json").write_text(json.dumps({"results": results, "passed": passed}, indent=1, default=float))
identity_records.to_csv(artifacts / "checks-identity-records.csv", index=False)
false_alarm.to_csv(artifacts / "checks-false-alarm.csv", index=False)
passed

In [ ]:
assert all(passed.values()), passed